# Finance Business Partner View: Workforce Cost & Budget Risk

This notebook creates scenario-based features for Finance decisions about permanent and contract staffing.

It outputs job-level workforce-cost and vacancy-budget-exposure features, industry-level risk summaries, and matched permanent-versus-contract economics.

> **Interpretation guardrail:** `metadata_expiryDate - metadata_originalPostingDate` is a posting-window proxy, not observed time-to-fill. Vacancy-budget exposure estimates the financial exposure associated with an unfilled approved role; it is not a recorded cash loss.

In [8]:
from pathlib import Path
import ast

import numpy as np
import pandas as pd

pd.set_option("display.float_format", "{:,.2f}".format)

# Planning inputs, not facts contained in the job-posting data.
PERMANENT_EMPLOYER_BURDEN_RATE = 0.17
CONTRACT_EMPLOYER_BURDEN_RATE = 0.02
CONTRACT_AGENCY_PREMIUM_RATE = 0.12
DAYS_PER_MONTH = 30.3
COST_NEUTRAL_TOLERANCE_RATE = 0.03

scenario = pd.Series({
    "permanent_employer_burden_rate": PERMANENT_EMPLOYER_BURDEN_RATE,
    "contract_employer_burden_rate": CONTRACT_EMPLOYER_BURDEN_RATE,
    "contract_agency_premium_rate": CONTRACT_AGENCY_PREMIUM_RATE,
    "cost_neutral_tolerance_rate": COST_NEUTRAL_TOLERANCE_RATE,
})
display(scenario.to_frame("value"))

,value
permanent_employer_burden_rate,0.17
contract_employer_burden_rate,0.02
contract_agency_premium_rate,0.12
cost_neutral_tolerance_rate,0.03


In [13]:
clean_path = Path("../data/processed/SGJobData_cleaned.csv")
feature_dir = Path("../data/features")
feature_dir.mkdir(parents=True, exist_ok=True)

use_columns = [
    "categories", "employmentTypes", "metadata_jobPostId",
    "metadata_originalPostingDate", "metadata_newPostingDate",
    "metadata_expiryDate", "numberOfVacancies", "positionLevels",
    "salary_minimum", "salary_maximum", "average_salary", "salary_type", "title",
]
jobs = pd.read_csv(
    clean_path,
    usecols=use_columns,
    parse_dates=[
        "metadata_originalPostingDate", "metadata_newPostingDate", "metadata_expiryDate",
    ],
)

def primary_industry(value):
    """Return the first advertised category as a stable industry reporting dimension."""
    try:
        categories = ast.literal_eval(value)
    except (SyntaxError, ValueError) as error:
        raise ValueError(f"Invalid categories value: {str(value)[:200]}") from error

    if not isinstance(categories, list) or not categories:
        return "Unclassified"
    return categories[0].get("category", "Unclassified")

employment_cohort = {
    "Permanent": "Permanent", "Full Time": "Permanent",
    "Contract": "Contract", "Temporary": "Contract",
    "Freelance": "Contract", "Flexi-work": "Contract",
}
jobs["primary_industry"] = jobs["categories"].map(primary_industry)
jobs["employment_cohort"] = jobs["employmentTypes"].map(employment_cohort).fillna("Other")
jobs["salary_midpoint_monthly"] = jobs[["salary_minimum", "salary_maximum"]].mean(axis=1)

assert jobs["salary_midpoint_monthly"].ge(0).all(), "Salary midpoint cannot be negative."
assert jobs["salary_type"].eq("Monthly").all(), "Salary type must be monthly for this analysis."
display(jobs[["employmentTypes", "employment_cohort"]].value_counts().to_frame("postings"))
display(jobs.head())

,,postings
employmentTypes,employment_cohort,
Permanent,Permanent,458139
Full Time,Permanent,393352
Contract,Contract,139182
Part Time,Other,25431
Temporary,Contract,18241
Internship/Attachment,Other,6959
Freelance,Contract,2139
Flexi-work,Contract,1154


,categories,employmentTypes,metadata_expiryDate,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,numberOfVacancies,positionLevels,salary_maximum,salary_minimum,salary_type,title,average_salary,primary_industry,employment_cohort,salary_midpoint_monthly
0,"[{'id': 13, 'category': 'Environment / Health'...",Permanent,2023-05-08,MCF-2023-0252866,2023-04-08,2023-03-30,1,Executive,"2,800.00","2,000.00",Monthly,Food Technologist - Clementi | Entry Level | U...,"2,400.00",Environment / Health,Permanent,"2,400.00"
1,"[{'id': 21, 'category': 'Information Technolog...",Permanent,2023-05-08,MCF-2023-0273977,2023-04-08,2023-04-08,2,Executive,"5,500.00","4,000.00",Monthly,"Software Engineer (Fab Support) (Java, CIM, Up...","4,750.00",Information Technology,Permanent,"4,750.00"
2,"[{'id': 33, 'category': 'Repair and Maintenanc...",Full Time,2023-04-22,MCF-2023-0273994,2023-04-08,2023-04-08,1,Senior Executive,"4,600.00","3,800.00",Monthly,Senior Technician,"4,200.00",Repair and Maintenance,Permanent,"4,200.00"
3,"[{'id': 21, 'category': 'Information Technolog...",Permanent,2023-05-08,MCF-2023-0273991,2023-04-08,2023-04-08,1,Senior Executive,"10,000.00","5,000.00",Monthly,"Senior .NET Developer (.NET Core, MVC, MVVC, S...","7,500.00",Information Technology,Permanent,"7,500.00"
4,"[{'id': 2, 'category': 'Admin / Secretarial'}]",Full Time,2023-05-08,MCF-2023-0273976,2023-04-08,2023-04-08,3,Non-executive,"3,400.00","2,400.00",Monthly,Sales / Admin Cordinator,"2,900.00",Admin / Secretarial,Permanent,"2,900.00"


## Progressive Wage Model Review Flags

The source declares every salary as monthly. This review identifies postings that may belong to a Progressive Wage Model sector from their title and broad industry, but it does not infer worker eligibility, progression level, or a different salary frequency. Enter Finance-approved, effective-dated policy floors before treating a flag as a compliance conclusion.

In [ ]:
# Candidate matching is intentionally conservative. Broad categories alone are not a PWM eligibility decision.
pwm_sector_rules = [
    ("Cleaning", r"\b(?:cleaner|cleaning|housekeep(?:er|ing))\b", ["Environment / Health", "General Work", "Hospitality"]),
    ("Security", r"\b(?:security officer|security guard|security supervisor|guard)\b", ["Security and Investigation"]),
    ("Landscape", r"\b(?:landscap(?:e|ing)|gardener|horticultur)\b", ["Environment / Health", "General Work"]),
    ("Lift and escalator", r"\b(?:lift|escalator)\b", ["Repair and Maintenance", "Engineering"]),
    ("Retail", r"\b(?:retail|cashier|shop assistant)\b", ["Sales / Retail"]),
    ("Food services", r"\b(?:barista|waiter|waitress|kitchen|service crew|cook)\b", ["F&B"]),
    ("Waste management", r"\b(?:waste|refuse|recycl)\b", ["Environment / Health", "General Work"]),
]

jobs["pwm_sector_candidate"] = pd.NA
for sector, pattern, industries in pwm_sector_rules:
    match = (
        jobs["title"].str.contains(pattern, case=False, na=False, regex=True)
        & jobs["primary_industry"].isin(industries)
    )
    jobs.loc[match & jobs["pwm_sector_candidate"].isna(), "pwm_sector_candidate"] = sector

# Maintain this table with Finance/HR-approved MOM policy values by sector, role, and effective date.
# An empty value means that the dataset cannot support a below-floor conclusion for the posting.
pwm_monthly_floor_reference = pd.DataFrame(
    columns=["pwm_sector_candidate", "effective_from", "effective_to", "monthly_floor"]
)

jobs["pwm_monthly_floor"] = pd.Series(pd.NA, index=jobs.index, dtype="Float64")
jobs["pwm_salary_check_status"] = np.where(
    jobs["pwm_sector_candidate"].notna(),
    "policy_floor_not_configured",
    "not_a_pwm_sector_candidate",
)
jobs["salary_frequency_interpretation"] = "monthly_as_declared"
jobs["salary_frequency_review_required"] = False

print(f"Potential PWM-sector postings: {jobs['pwm_sector_candidate'].notna().sum():,}")
display(jobs.loc[jobs["pwm_sector_candidate"].notna(), [
    "title", "primary_industry", "salary_type", "salary_midpoint_monthly",
    "pwm_sector_candidate", "pwm_salary_check_status", "salary_frequency_interpretation",
]].head(10))

/tmp/ipykernel_25796/3767500866.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  match = jobs["title"].str.contains(pattern, case=False, na=False, regex=True)


Potential PWM-sector postings: 59,032


,title,primary_industry,salary_type,salary_midpoint_monthly,pwm_sector_candidate,pwm_salary_check_status,salary_frequency_interpretation
7,IT Security Engineer (Maritime/ Cloud Security),Security and Investigation,Monthly,"6,750.00",Security,policy_floor_not_configured,monthly_as_declared
11,Cook / Chef de Partie / Kitchen Assistant,F&B,Monthly,"2,750.00",Food services,policy_floor_not_configured,monthly_as_declared
21,"Brand & Marketing Manager (Concept Branding, F&B)",Marketing / Public Relations,Monthly,"4,500.00",Food services,policy_floor_not_configured,monthly_as_declared
23,Worknow: Barista/ Service Crew,Admin / Secretarial,Monthly,"3,000.00",Food services,policy_floor_not_configured,monthly_as_declared
24,Urgent!!! Assistant Property Leasing Manager (...,Real Estate / Property Management,Monthly,"5,000.00",Retail,policy_floor_not_configured,monthly_as_declared
38,"Cyber Security Engineer (Post Sales, Imperva, ...",Information Technology,Monthly,"5,000.00",Security,policy_floor_not_configured,monthly_as_declared
46,Network Security Infra Specialist,Information Technology,Monthly,"8,500.00",Security,policy_floor_not_configured,monthly_as_declared
49,Back of House (Luxury Retail - Support) - No s...,Admin / Secretarial,Monthly,"3,150.00",Retail,policy_floor_not_configured,monthly_as_declared
78,Assistant Manager / Restaurant (Urgent Role),Others,Monthly,"3,000.00",Food services,policy_floor_not_configured,monthly_as_declared
102,Security Operations Engineer (Ref 25248),Information Technology,Monthly,"5,500.00",Security,policy_floor_not_configured,monthly_as_declared


In [10]:
# Cap both salary tails only for planning metrics so malformed or exceptional advertisements
# do not dominate aggregate budget-risk rankings. The raw midpoint remains available.
SALARY_PLANNING_CAP_QUANTILE = 0.995
SALARY_PLANNING_FLOOR_QUANTILE = 1 - SALARY_PLANNING_CAP_QUANTILE
salary_planning_floor = jobs["salary_midpoint_monthly"].quantile(SALARY_PLANNING_FLOOR_QUANTILE)
salary_planning_cap = jobs["salary_midpoint_monthly"].quantile(SALARY_PLANNING_CAP_QUANTILE)
jobs["salary_low_outlier_flag"] = jobs["salary_midpoint_monthly"].lt(salary_planning_floor)
jobs["salary_high_outlier_flag"] = jobs["salary_midpoint_monthly"].gt(salary_planning_cap)
jobs["planning_salary_midpoint_monthly"] = jobs["salary_midpoint_monthly"].clip(
    lower=salary_planning_floor,
    upper=salary_planning_cap,
)

print(f"Planning salary floor at p{SALARY_PLANNING_FLOOR_QUANTILE:.1%}: ${salary_planning_floor:,.2f}")
print(f"Planning salary cap at p{SALARY_PLANNING_CAP_QUANTILE:.1%}: ${salary_planning_cap:,.2f}")
print(f"Flagged low salary outliers: {jobs['salary_low_outlier_flag'].sum():,}")
print(f"Flagged high salary outliers: {jobs['salary_high_outlier_flag'].sum():,}")

Planning salary floor at p0.5%: $15.00
Planning salary cap at p99.5%: $19,250.00
Flagged low salary outliers: 5,181
Flagged high salary outliers: 5,212


In [11]:
jobs["posting_window_days"] = (
    jobs["metadata_expiryDate"] - jobs["metadata_originalPostingDate"]
).dt.days.clip(lower=0)
jobs["new_posting_lag_days"] = (
    jobs["metadata_newPostingDate"] - jobs["metadata_originalPostingDate"]
).dt.days.clip(lower=0)

jobs["employer_burden_rate"] = np.select(
    [jobs["employment_cohort"].eq("Permanent"), jobs["employment_cohort"].eq("Contract")],
    [PERMANENT_EMPLOYER_BURDEN_RATE, CONTRACT_EMPLOYER_BURDEN_RATE],
    default=0.0,
)
jobs["agency_premium_rate"] = np.where(
    jobs["employment_cohort"].eq("Contract"), CONTRACT_AGENCY_PREMIUM_RATE, 0.0
)
jobs["total_cost_multiplier"] = 1 + jobs["employer_burden_rate"] + jobs["agency_premium_rate"]
jobs["loaded_monthly_cost_per_head"] = (
    jobs["planning_salary_midpoint_monthly"] * jobs["total_cost_multiplier"]
)
jobs["loaded_monthly_cost_for_vacancies"] = (
    jobs["loaded_monthly_cost_per_head"] * jobs["numberOfVacancies"]
)
jobs["daily_loaded_cost_for_vacancies"] = (
    jobs["loaded_monthly_cost_for_vacancies"] / DAYS_PER_MONTH
)
jobs["vacancy_budget_exposure"] = (
    jobs["daily_loaded_cost_for_vacancies"] * jobs["posting_window_days"]
)
jobs["vacancy_exposure_per_opening"] = (
    jobs["vacancy_budget_exposure"] / jobs["numberOfVacancies"].replace(0, np.nan)
)

assert jobs["posting_window_days"].ge(0).all(), "Posting window cannot be negative."
assert jobs["vacancy_budget_exposure"].ge(0).all(), "Budget exposure cannot be negative."
display(jobs[["posting_window_days", "loaded_monthly_cost_per_head", "vacancy_budget_exposure"]].describe())

,posting_window_days,loaded_monthly_cost_per_head,vacancy_budget_exposure
count,"1,044,597.00","1,044,597.00","1,044,597.00"
mean,28.39,"5,416.43","13,058.77"
std,18.03,"3,383.95","295,448.10"
min,1.00,15.00,3.47
25%,30.00,"3,393.00","3,475.25"
50%,30.00,"4,446.00","5,792.08"
75%,30.00,"6,435.00","11,468.32"
max,335.00,"22,522.50","137,805,222.77"


In [12]:
finance_jobs = jobs.drop(columns=["categories"]).copy()
finance_jobs.to_parquet(feature_dir / "job_finance_features.parquet", index=False)

industry_budget_risk = (
    finance_jobs.groupby("primary_industry", as_index=False)
    .agg(
        postings=("metadata_jobPostId", "nunique"),
        vacancies=("numberOfVacancies", "sum"),
        median_posting_window_days=("posting_window_days", "median"),
        p90_posting_window_days=("posting_window_days", lambda values: values.quantile(0.90)),
        total_vacancy_budget_exposure=("vacancy_budget_exposure", "sum"),
        median_exposure_per_opening=("vacancy_exposure_per_opening", "median"),
    )
    .sort_values("total_vacancy_budget_exposure", ascending=False)
)
industry_budget_risk["budget_exposure_rank"] = range(1, len(industry_budget_risk) + 1)
industry_budget_risk.to_csv(feature_dir / "industry_budget_risk.csv", index=False)

MIN_COHORT_POSTINGS = 30
cohort_benchmarks = (
    finance_jobs.query("employment_cohort in ['Permanent', 'Contract']")
    .groupby(["primary_industry", "positionLevels", "employment_cohort"], as_index=False)
    .agg(
        postings=("metadata_jobPostId", "nunique"),
        median_loaded_monthly_cost=("loaded_monthly_cost_per_head", "median"),
        median_posting_window_days=("posting_window_days", "median"),
    )
)

conversion_economics = (
    cohort_benchmarks.pivot(
        index=["primary_industry", "positionLevels"],
        columns="employment_cohort",
        values=["postings", "median_loaded_monthly_cost", "median_posting_window_days"],
    )
    .dropna(subset=[("median_loaded_monthly_cost", "Permanent"), ("median_loaded_monthly_cost", "Contract")])
)
conversion_economics.columns = [f"{metric}_{cohort.lower()}" for metric, cohort in conversion_economics.columns]
conversion_economics = conversion_economics.reset_index()
conversion_economics = conversion_economics.query(
    "postings_contract >= @MIN_COHORT_POSTINGS and postings_permanent >= @MIN_COHORT_POSTINGS"
).copy()
conversion_economics["monthly_cost_delta_contract_minus_permanent"] = (
    conversion_economics["median_loaded_monthly_cost_contract"]
    - conversion_economics["median_loaded_monthly_cost_permanent"]
)
conversion_economics["contract_cost_delta_rate"] = (
    conversion_economics["monthly_cost_delta_contract_minus_permanent"]
    / conversion_economics["median_loaded_monthly_cost_permanent"]
)
conversion_economics["conversion_decision"] = np.select(
    [
        conversion_economics["contract_cost_delta_rate"].le(-COST_NEUTRAL_TOLERANCE_RATE),
        conversion_economics["contract_cost_delta_rate"].ge(COST_NEUTRAL_TOLERANCE_RATE),
    ],
    ["Savings", "Cost premium"],
    default="Wash",
)
conversion_economics = conversion_economics.sort_values("monthly_cost_delta_contract_minus_permanent")
conversion_economics.to_csv(feature_dir / "permanent_contract_conversion_economics.csv", index=False)

print(f"Conversion comparisons meeting the {MIN_COHORT_POSTINGS}-posting threshold: {len(conversion_economics):,}")
display(industry_budget_risk.head(10))
display(conversion_economics.head(10))

Conversion comparisons meeting the 30-posting threshold: 237


,primary_industry,postings,vacancies,median_posting_window_days,p90_posting_window_days,total_vacancy_budget_exposure,median_exposure_per_opening,budget_exposure_rank
20,Information Technology,100142,236667,30.00,30.00,"1,721,096,714.12","6,371.29",1
1,Admin / Secretarial,102719,229399,30.00,30.00,"1,091,321,457.81","3,011.88",2
10,Engineering,99675,190675,30.00,30.00,"990,355,428.58","4,344.06",3
7,Customer Service,64865,240390,30.00,30.00,"918,763,997.61","3,185.64",4
14,F&B,59678,244260,30.00,30.00,"892,420,224.25","3,533.17",5
0,Accounting / Auditing / Taxation,78648,164882,30.00,30.00,"840,608,654.07","3,781.19",6
5,Building and Construction,74014,159855,30.00,30.00,"838,106,244.70","4,344.06",7
4,Banking and Finance,46635,110853,30.00,30.00,"768,456,448.71","6,081.68",8
17,Healthcare / Pharmaceutical,33229,168524,30.00,30.00,"743,924,387.67","3,996.53",9
6,Consulting,24187,100884,30.00,30.00,"642,796,162.85","5,270.79",10


,primary_industry,positionLevels,postings_contract,postings_permanent,median_loaded_monthly_cost_contract,median_loaded_monthly_cost_permanent,median_posting_window_days_contract,median_posting_window_days_permanent,monthly_cost_delta_contract_minus_permanent,contract_cost_delta_rate,conversion_decision
40,Banking and Finance,Middle Management,714.00,"1,812.00","9,120.00","15,795.00",30.00,30.00,"-6,675.00",-0.42,Savings
178,Human Resources,Senior Management,123.00,528.00,"6,840.00","11,700.00",7.00,30.00,"-4,860.00",-0.42,Savings
44,Banking and Finance,Senior Management,96.00,"1,840.00","12,967.50","17,023.50",30.00,30.00,"-4,056.00",-0.24,Savings
26,Advertising / Media,Senior Management,31.00,411.00,"9,405.00","12,285.00",30.00,30.00,"-2,880.00",-0.23,Savings
42,Banking and Finance,Professional,"2,058.00","5,751.00","8,550.00","11,407.50",30.00,30.00,"-2,857.50",-0.25,Savings
194,Insurance,Professional,69.00,351.00,"6,840.00","9,652.50",30.00,30.00,"-2,812.50",-0.29,Savings
62,Consulting,Senior Management,70.00,"1,117.00","11,827.50","14,625.00",30.00,30.00,"-2,797.50",-0.19,Savings
8,Accounting / Auditing / Taxation,Senior Management,80.00,"1,852.00","9,120.00","11,700.00",30.00,30.00,"-2,580.00",-0.22,Savings
143,General Management,Senior Management,90.00,"1,708.00","10,374.00","12,870.00",21.00,30.00,"-2,496.00",-0.19,Savings
187,Information Technology,Senior Management,249.00,"1,516.00","11,115.00","13,455.00",30.00,30.00,"-2,340.00",-0.17,Savings
